In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_tavily import TavilySearch       # langchain-tavily 패키지
from langchain.agents import create_agent
import ast
import operator

In [3]:
# 1. 도구 정의: 웹 검색
search_tool = TavilySearch(max_results=3)

# 2. 도구 정의: 안전한 계산기
SAFE_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        op = SAFE_OPERATORS.get(type(node.op))
        if op is None:
            raise ValueError(f"지원하지 않는 연산자: {node.op}")
        return op(_safe_eval(node.left), _safe_eval(node.right))
    raise ValueError(f"허용되지 않는 표현식: {node}")

In [4]:
@tool
def calculate(expression: str) -> str:
    """사칙연산과 거듭제곱만 허용하는 안전한 계산기입니다. 예: '1234 * 5678'"""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"


In [7]:
# 3. 에이전트 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=[search_tool, calculate],
    system_prompt="""당신은 웹 검색과 계산을 도와주는 AI 어시스턴트입니다.
            규칙:
            - 동일하거나 유사한 검색 쿼리를 반복하지 마세요.
            - 검색 결과가 없으면 즉시 그 사실을 사용자에게 알리세요.
            - 최대 3번 검색 후 반드시 최종 답변을 제공하세요.""",
)


In [8]:
# 4. 테스트 실행
print("=== 테스트 1: 검색만 사용 ===")
result1 = agent.invoke({
    "messages": [{"role": "user", "content": "2024년 한국 GDP는 얼마야?"}]
})
print(result1["messages"][-1].content)

=== 테스트 1: 검색만 사용 ===
2024년 한국의 GDP는 약 1.87조 달러(1,875,388 백만 달러)로 예상됩니다. 이는 한국 경제가 세계에서 12위에 해당하는 규모입니다. 또한, 1인당 GDP는 약 36,024 달러로 추정되고 있습니다. 

더 자세한 정보는 [여기](https://www.kedglobal.com/economy/newsView/ked202501230002)에서 확인하실 수 있습니다.


In [16]:
for msg in result1["messages"]:
    name = msg.__class__.__name__
    # 툴 호출 정보 추출
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        for tc in tool_calls:
            print(f"[{name}] 도구 호출: {tc['name']}({tc['args']})")
    else:
        content = str(msg.content)[:120] if msg.content else ""
        print(f"[{name}] {content}")

[HumanMessage] 2024년 한국 GDP는 얼마야?
[AIMessage] 도구 호출: tavily_search({'query': '2024 South Korea GDP forecast'})
[ToolMessage] {"query": "2024 South Korea GDP forecast", "response_time": 0.78, "follow_up_questions": null, "answer": null, "images":
[AIMessage] 도구 호출: tavily_search({'query': '2024 South Korea GDP estimate'})
[ToolMessage] {"query": "2024 South Korea GDP estimate", "response_time": 0.73, "follow_up_questions": null, "answer": null, "images":
[AIMessage] 도구 호출: tavily_search({'query': '2024 South Korea GDP total value'})
[ToolMessage] {"query": "2024 South Korea GDP total value", "response_time": 1.0, "follow_up_questions": null, "answer": null, "images
[AIMessage] 2024년 한국의 GDP는 약 1.87조 달러(1,875,388 백만 달러)로 예상됩니다. 이는 한국 경제가 세계에서 12위에 해당하는 규모입니다. 또한, 1인당 GDP는 약 36,024 달러로 추정되고 있습니다. 


In [9]:
print("\n=== 테스트 2: 계산기만 사용 ===")
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "1234 곱하기 5678은 얼마야?"}]
})
print(result2["messages"][-1].content)


=== 테스트 2: 계산기만 사용 ===
1234 곱하기 5678은 7,006,652입니다.


In [10]:
print("\n=== 테스트 3: 검색 + 계산기 모두 사용 ===")
result3 = agent.invoke({
    "messages": [{"role": "user", "content": "2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?"}]
})
print(result3["messages"][-1].content)


=== 테스트 3: 검색 + 계산기 모두 사용 ===
2024년 한국의 GDP는 약 1.9 조 달러(1,875,388 백만 달러)입니다. 이 금액을 1300으로 나누면 약 1,442,307.69가 됩니다.


In [11]:
# result3 = agent.invoke({
#     "messages": [{"role": "user", "content": "2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?"}]
# })

for msg in result3["messages"]:
    name = msg.__class__.__name__
    # 툴 호출 정보 추출
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        for tc in tool_calls:
            print(f"[{name}] 도구 호출: {tc['name']}({tc['args']})")
    else:
        content = str(msg.content)[:120] if msg.content else ""
        print(f"[{name}] {content}")

[HumanMessage] 2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?
[AIMessage] 도구 호출: tavily_search({'query': '2024 South Korea GDP'})
[ToolMessage] {"query": "2024 South Korea GDP", "response_time": 0.63, "follow_up_questions": null, "answer": null, "images": [], "res
[AIMessage] 도구 호출: calculate({'expression': '1875000000/1300'})
[ToolMessage] 1442307.6923076923
[AIMessage] 2024년 한국의 GDP는 약 1.9 조 달러(1,875,388 백만 달러)입니다. 이 금액을 1300으로 나누면 약 1,442,307.69가 됩니다.


In [12]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?"}]},
    stream_mode="updates",   # 각 노드 업데이트마다 출력
):
    for node_name, node_output in step.items():
        print(f"\n{'='*40}")
        print(f"노드: {node_name}")
        for msg in node_output.get("messages", []):
            name = msg.__class__.__name__
            tool_calls = getattr(msg, "tool_calls", [])
            if tool_calls:
                for tc in tool_calls:
                    print(f"  도구 호출: {tc['name']}({tc['args']})")
            else:
                print(f"  [{name}] {str(msg.content)[:120]}")


노드: model
  도구 호출: tavily_search({'query': '2024 South Korea GDP'})

노드: tools
  [ToolMessage] {"query": "2024 South Korea GDP", "response_time": 0.63, "follow_up_questions": null, "answer": null, "images": [], "res

노드: model
  도구 호출: calculate({'expression': '1875000/1300'})

노드: tools
  [ToolMessage] 1442.3076923076924

노드: model
  [AIMessage] 2024년 한국의 GDP는 약 1.9 조 달러(1,875,388 백만 달러)입니다. 이 금액을 1300으로 나누면 약 1442.31입니다.


In [13]:
# 3. 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "2024년 한국 GDP는 얼마야"}]
})

print(result["messages"][-1].content)

2024년 한국의 GDP는 약 1.87조 달러(1,875,388 백만 달러)로 예상됩니다. 이는 한국 경제가 세계에서 13번째로 큰 규모임을 나타냅니다. 또한, 1인당 GDP는 약 36,024 달러로 추정되고 있습니다. 

더 자세한 정보는 [여기](https://www.kedglobal.com/economy/newsView/ked202501230002)에서 확인하실 수 있습니다.


In [14]:
for msg in result["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content[:80] if msg.content else '(tool call)'}")

[HumanMessage] 2024년 한국 GDP는 얼마야
[AIMessage] (tool call)
[ToolMessage] {"query": "2024 South Korea GDP forecast", "response_time": 0.78, "follow_up_que
[AIMessage] (tool call)
[ToolMessage] {"query": "2024 South Korea GDP estimate", "response_time": 0.73, "follow_up_que
[AIMessage] (tool call)
[ToolMessage] {"query": "2024 South Korea GDP total value", "response_time": 1.0, "follow_up_q
[AIMessage] 2024년 한국의 GDP는 약 1.87조 달러(1,875,388 백만 달러)로 예상됩니다. 이는 한국 경제가 세계에서 13번째로 큰 규모임을 나


In [15]:
result

{'messages': [HumanMessage(content='2024년 한국 GDP는 얼마야', additional_kwargs={}, response_metadata={}, id='e1a3d5ad-192b-41e6-a04b-63ebf56a637b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 1313, 'total_tokens': 1334, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b9de628d4e', 'id': 'chatcmpl-DRv7T0Or8Fq26anhkqHRZ7kNyu4l4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d66d5-b313-7791-92b0-bf2feacaab53-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2024 South Korea GDP forecast'}, 'id': 'call_ePqCivyYodHI2hLR9RgwKIM0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t